# 02 — Limpieza y Transformación de Efectos Secundarios por Componentes

Este notebook ejecuta el pipeline completo de preparación de datos para el
Enfoque 3: Efectos Secundarios por Componentes.

Tareas:
- Limpiar el dataset: eliminar duplicados, parsear componentes y efectos.
- Transformar: expandir a nivel componente × efecto, construir crosstab y crosstab normalizado.
- Guardar todos los artefactos en `data/processed/`.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.load_data import load_medicine_data
from src.enfoque_03_efectos_secundarios_componentes.cleaning import run_cleaning_pipeline

# 1. Cargar datos raw (sin descargar)
df_raw = load_medicine_data(download_if_missing=True)
print(f"Shape raw: {df_raw.shape}")

# 2. Ejecutar limpieza completa
df_clean = run_cleaning_pipeline(df_raw, save=True)

# 3. Verificar resultado
df_clean[["Medicine Name", "componentes", "efectos_secundarios",
          "n_componentes", "n_efectos",
          "anomalia_componentes", "anomalia_efectos"]].head(10)

Shape raw: (11825, 9)
INICIO DEL PIPELINE DE LIMPIEZA  [Enfoque 3]
[eliminar_duplicados] Filas antes: 11825 | Eliminados: 84 | Filas después: 11741
[añadir_columnas_derivadas] Columnas generadas: componentes, efectos_secundarios, manufacturer, n_componentes, n_efectos
[añadir_columnas_derivadas] Distribución por número de componentes:
    1 componente(s)  ->  7019 medicamentos
    2 componente(s)  ->  3569 medicamentos
    3 componente(s)  ->   929 medicamentos
    4 componente(s)  ->   147 medicamentos
    5 componente(s)  ->    51 medicamentos
    6 componente(s)  ->    16 medicamentos
    7 componente(s)  ->     7 medicamentos
    8 componente(s)  ->     2 medicamentos
    9 componente(s)  ->     1 medicamentos
[flag_anomalias] Registros sin componentes: 0
[flag_anomalias] Registros sin efectos secundarios: 0
-------------------------------------------------------
Forma final del DataFrame: (11741, 16)
[run_cleaning_pipeline] CSV limpio guardado en: /home/yvl/Documentos/Duoc/Trabajo

,Medicine Name,componentes,efectos_secundarios,n_componentes,n_efectos,anomalia_componentes,anomalia_efectos
0,Avastin 400mg Injection,[bevacizumab],"[rectal bleeding, taste change, headache, nose...",1,9,False,False
1,Augmentin 625 Duo Tablet,"[amoxycillin, clavulanic acid]","[vomiting, nausea, diarrhea, mucocutaneous can...",2,4,False,False
2,Azithral 500 Tablet,[azithromycin],"[nausea, abdominal pain, diarrhea]",1,3,False,False
3,Ascoril LS Syrup,"[ambroxol, levosalbutamol, guaifenesin]","[nausea, vomiting, diarrhea, upset stomach, st...",3,14,False,False
4,Aciloc 150 Tablet,[ranitidine],"[headache, diarrhea, gastrointestinal disturba...",1,3,False,False
5,Allegra 120mg Tablet,[fexofenadine],"[headache, drowsiness, dizziness, nausea]",1,4,False,False
6,Avil 25 Tablet,[pheniramine],[sedation],1,1,False,False
7,Aricep 5 Tablet,[donepezil],"[common cold, urinary incontinence, rash, naus...",1,8,False,False
8,Amoxyclav 625 Tablet,"[amoxycillin, clavulanic acid]","[vomiting, nausea, diarrhea, mucocutaneous can...",2,4,False,False
9,Atarax 25mg Tablet,[hydroxyzine],"[sedation, nausea, vomiting, upset stomach, co...",1,5,False,False


In [2]:
from src.enfoque_03_efectos_secundarios_componentes.transform import run_transform_pipeline

# df_clean ya disponible de la celda anterior
df_long, crosstab, crosstab_norm = run_transform_pipeline(
    df_clean, min_observaciones=5, top_n_efectos=30, save=True
)

# Verificar cada resultado
print("\n--- df_long (componente × efecto expandido) ---")
print(df_long[["componentes", "efectos_secundarios"]].head(5))

print("\n--- crosstab (frecuencias absolutas) ---")
print(crosstab.iloc[:5, :5])

print("\n--- crosstab_norm (normalizado por fila) ---")
print(crosstab_norm.iloc[:5, :5].round(3))

INICIO DEL PIPELINE DE TRANSFORMACIONES  [Enfoque 3]
[explotar_todo] Filas antes: 11741 -> después: 123469
[explotar_todo] Pares únicos (componente, efecto): 11005
[construir_crosstab] Componentes incluidos: 875 (min_obs=5)
[construir_crosstab] Efectos incluidos   : 761
[construir_crosstab] Forma de la tabla   : 875×761
[normalizar_crosstab] Tabla normalizada por fila: 875×761
-------------------------------------------------------
df_long shape       : (123469, 16)
crosstab shape      : (875, 761)
crosstab_norm shape : (875, 761)
[run_transform_pipeline] Archivos guardados en: /home/yvl/Documentos/Duoc/TrabajoSCY/SCY1101-Exp1-Drug-Data-Analysis/data/processed

--- df_long (componente × efecto expandido) ---
   componentes efectos_secundarios
0  bevacizumab     rectal bleeding
1  bevacizumab        taste change
2  bevacizumab            headache
3  bevacizumab          nosebleeds
4  bevacizumab           back pain

--- crosstab (frecuencias absolutas) ---
efectos_secundarios  abdominal